# LightGlue Demo

In [ ]:
from pathlib import Path
from lightglue import LightGlue, SuperPoint, DISK
from lightglue.utils import load_image, rbd
from lightglue import viz2d
import torch
import matplotlib.pyplot as plt
import random

## Load extractor and matcher module
In this example we use SuperPoint features combined with LightGlue.

In [7]:
torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 'mps', 'cpu'

extractor = SuperPoint(max_num_keypoints=2048).eval().to(device)  # load the extractor
matcher = LightGlue(features="superpoint").eval().to(device)

## Easy example
The top image shows the matches, while the bottom image shows the point pruning across layers. In this case, LightGlue prunes a few points with occlusions, but is able to stop the context aggregation after 4/9 layers.

In [ ]:
# Parameters
match_confidence_treshold = 0.8
number_of_matches = 20



claims_directory = "/nfs_disk/datasets/goldset_v2/"
claim = random.choice([x for x in Path(claims_directory).iterdir() if x.is_dir()])

images = [x for x in Path(claim).iterdir()]

combinations = []
for idx_1 in range(len(images)):
    for idx_2 in range(idx_1+1, len(images)):
        combinations.append((images[idx_1], images[idx_2]))

print(f"Claim {str(claim).replace(f"{claims_directory}claim_", "")}")
print(f"Computing {len(combinations)} image pairs with {len(images)} images...\n")

for pair in combinations:
    image0 = load_image(pair[0])
    image1 = load_image(pair[1])

    feats0 = extractor.extract(image0.to(device))
    feats1 = extractor.extract(image1.to(device))
    matches01 = matcher({"image0": feats0, "image1": feats1})

    feats0, feats1, matches01 = [
        rbd(x) for x in [feats0, feats1, matches01]
    ]

    kpts0, kpts1, matches, scores = feats0["keypoints"], feats1["keypoints"], matches01["matches"], matches01["scores"]

    valid_mask = scores > match_confidence_treshold
    valid_matches = matches[valid_mask]
    valid_scores = scores[valid_mask]

    m_kpts0 = kpts0[valid_matches[:, 0]]
    m_kpts1 = kpts1[valid_matches[:, 1]]


    if len(valid_scores) > number_of_matches and valid_scores.max() >= match_confidence_treshold:
        axes = viz2d.plot_images([image0, image1])
        
        viz2d.plot_matches(
            m_kpts0, m_kpts1, 
            color=[(1 - s.item(), s.item(), 0) for s in valid_scores],
            lw=1
        )

        viz2d.add_text(0, f"Matches: {len(valid_matches)}\n", fs=20)
        
        plt.show()